In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that multiplies a sparse matrix <code>A</code> of dimensions <code>M</code> &times; <code>N</code>
  by a dense matrix <code>B</code> of dimensions <code>N</code> &times; <code>K</code>, producing a dense output matrix
  <code>C</code> of dimensions <code>M</code> &times; <code>K</code>.
  All matrices are stored in row-major order using 32-bit floats.
  The matrix <code>A</code> is approximately 60&ndash;70% sparse (i.e., 60&ndash;70% of elements are zero),
  and <code>nnz</code> gives the number of non-zero elements in <code>A</code>.
</p>

<p>
  Mathematically, the operation is defined as:
  $$
  C_{ij} = \sum_{k=0}^{N-1} A_{ik} \cdot B_{kj} \quad \text{for} \quad i = 0, \ldots, M-1,\; j = 0, \ldots, K-1
  $$
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only GPU native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in matrix <code>C</code></li>
</ul>

<h2>Example</h2>
<p>
Input:<br>
Matrix $A$ ($3 \times 4$):
$$
\begin{bmatrix}
2.0 & 0.0 & 0.0 & 1.0 \\
0.0 & 3.0 & 0.0 & 0.0 \\
0.0 & 0.0 & 4.0 & 0.0
\end{bmatrix}
$$
Matrix $B$ ($4 \times 2$):
$$
\begin{bmatrix}
1.0 & 2.0 \\
3.0 & 4.0 \\
5.0 & 6.0 \\
7.0 & 8.0
\end{bmatrix}
$$
Output:<br>
Matrix $C$ ($3 \times 2$):
$$
\begin{bmatrix}
9.0 & 12.0 \\
9.0 & 12.0 \\
20.0 & 24.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 8,192</li>
  <li>All values in <code>A</code> and <code>B</code> are 32-bit floats in the range [&minus;10, 10]</li>
  <li>The matrix <code>A</code> is approximately 60&ndash;70% sparse</li>
  <li>Performance is measured with <code>M</code> = 4,096, <code>N</code> = 2,048, <code>K</code> = 512</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, B, C are device pointers
extern "C" void solve(const float* A, const float* B, float* C, int M, int N, int K, int nnz) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor,
    B: cute.Tensor,
    C: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
    nnz: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, M: int, N: int, K: int, nnz: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# A, B, C are device pointers
@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    C: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
    nnz: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, C are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, M: int, N: int, K: int, nnz: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# A, B, C are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, M: int, N: int, K: int, nnz: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Sparse Matrix-Dense Matrix Multiplication",
            atol=1e-03,
            rtol=1e-03,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        A: torch.Tensor,
        B: torch.Tensor,
        C: torch.Tensor,
        M: int,
        N: int,
        K: int,
        nnz: int,
    ):
        if A.shape == (M * N,):
            A_matrix = A.view(M, N)
        elif A.shape == (M, N):
            A_matrix = A
        else:
            raise AssertionError(
                f"A.shape {A.shape} does not match expected {(M * N,)} or {(M, N)}"
            )
        if B.shape == (N * K,):
            B_matrix = B.view(N, K)
        elif B.shape == (N, K):
            B_matrix = B
        else:
            raise AssertionError(
                f"B.shape {B.shape} does not match expected {(N * K,)} or {(N, K)}"
            )
        assert C.shape == (M, K) or C.shape == (
            M * K,
        ), f"C.shape {C.shape} does not match expected {(M, K)} or {(M * K,)}"
        assert A_matrix.dtype == torch.float32
        assert B_matrix.dtype == torch.float32
        assert A_matrix.device.type == "cuda"
        assert B_matrix.device.type == "cuda"
        assert C.device.type == "cuda"
        result = torch.matmul(A_matrix, B_matrix)
        C.copy_(result.view(C.shape))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_float), "in"),
            "B": (ctypes.POINTER(ctypes.c_float), "in"),
            "C": (ctypes.POINTER(ctypes.c_float), "out"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
            "K": (ctypes.c_int, "in"),
            "nnz": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        A = torch.tensor(
            [
                [2.0, 0.0, 0.0, 1.0],
                [0.0, 3.0, 0.0, 0.0],
                [0.0, 0.0, 4.0, 0.0],
            ],
            device="cuda",
            dtype=dtype,
        )
        B = torch.tensor(
            [
                [1.0, 2.0],
                [3.0, 4.0],
                [5.0, 6.0],
                [7.0, 8.0],
            ],
            device="cuda",
            dtype=dtype,
        )
        C = torch.empty((3, 2), device="cuda", dtype=dtype)
        return {
            "A": A,
            "B": B,
            "C": C,
            "M": 3,
            "N": 4,
            "K": 2,
            "nnz": 4,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # edge_1x1x1
        tests.append(
            {
                "A": torch.tensor([[3.0]], device="cuda", dtype=dtype),
                "B": torch.tensor([[2.0]], device="cuda", dtype=dtype),
                "C": torch.empty((1, 1), device="cuda", dtype=dtype),
                "M": 1,
                "N": 1,
                "K": 1,
                "nnz": 1,
            }
        )

        # edge_2x2_k1_spmv_like
        tests.append(
            {
                "A": torch.tensor([[1.0, 0.0], [0.0, 2.0]], device="cuda", dtype=dtype),
                "B": torch.tensor([[3.0], [4.0]], device="cuda", dtype=dtype),
                "C": torch.empty((2, 1), device="cuda", dtype=dtype),
                "M": 2,
                "N": 2,
                "K": 1,
                "nnz": 2,
            }
        )

        # edge_zero_matrix
        tests.append(
            {
                "A": torch.zeros((3, 3), device="cuda", dtype=dtype),
                "B": torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], device="cuda", dtype=dtype),
                "C": torch.empty((3, 2), device="cuda", dtype=dtype),
                "M": 3,
                "N": 3,
                "K": 2,
                "nnz": 0,
            }
        )

        # edge_identity_a
        tests.append(
            {
                "A": torch.eye(4, device="cuda", dtype=dtype),
                "B": torch.tensor(
                    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0], [10.0, 11.0, 12.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "C": torch.empty((4, 3), device="cuda", dtype=dtype),
                "M": 4,
                "N": 4,
                "K": 3,
                "nnz": 4,
            }
        )

        # power_of_2_16x16x8
        M, N, K = 16, 16, 8
        A_dense = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-2.0, 2.0)
        mask = torch.rand((M, N), device="cuda") > 0.65
        A_sparse = A_dense * mask
        tests.append(
            {
                "A": A_sparse,
                "B": torch.empty((N, K), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((M, K), device="cuda", dtype=dtype),
                "M": M,
                "N": N,
                "K": K,
                "nnz": int(mask.sum().item()),
            }
        )

        # power_of_2_64x32x16
        M, N, K = 64, 32, 16
        A_dense = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-3.0, 3.0)
        mask = torch.rand((M, N), device="cuda") > 0.70
        A_sparse = A_dense * mask
        tests.append(
            {
                "A": A_sparse,
                "B": torch.empty((N, K), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((M, K), device="cuda", dtype=dtype),
                "M": M,
                "N": N,
                "K": K,
                "nnz": int(mask.sum().item()),
            }
        )

        # non_power_of_2_negative_values
        M, N, K = 30, 50, 20
        A_dense = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-5.0, 5.0)
        mask = torch.rand((M, N), device="cuda") > 0.65
        A_sparse = A_dense * mask
        tests.append(
            {
                "A": A_sparse,
                "B": torch.empty((N, K), device="cuda", dtype=dtype).uniform_(-3.0, 3.0),
                "C": torch.empty((M, K), device="cuda", dtype=dtype),
                "M": M,
                "N": N,
                "K": K,
                "nnz": int(mask.sum().item()),
            }
        )

        # non_power_of_2_255x100x33
        M, N, K = 255, 100, 33
        A_dense = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-2.0, 2.0)
        mask = torch.rand((M, N), device="cuda") > 0.70
        A_sparse = A_dense * mask
        tests.append(
            {
                "A": A_sparse,
                "B": torch.empty((N, K), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((M, K), device="cuda", dtype=dtype),
                "M": M,
                "N": N,
                "K": K,
                "nnz": int(mask.sum().item()),
            }
        )

        # realistic_1000x500x64
        M, N, K = 1000, 500, 64
        A_dense = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        mask = torch.rand((M, N), device="cuda") > 0.65
        A_sparse = A_dense * mask
        tests.append(
            {
                "A": A_sparse,
                "B": torch.empty((N, K), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((M, K), device="cuda", dtype=dtype),
                "M": M,
                "N": N,
                "K": K,
                "nnz": int(mask.sum().item()),
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        M = 4096
        N = 2048
        K = 512
        A_dense = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        mask = torch.rand((M, N), device="cuda") > 0.65
        A_sparse = A_dense * mask
        nnz = int(mask.sum().item())
        B = torch.empty((N, K), device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        C = torch.empty((M, K), device="cuda", dtype=dtype)
        return {
            "A": A_sparse,
            "B": B,
            "C": C,
            "M": M,
            "N": N,
            "K": K,
            "nnz": nnz,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
